In [2]:
!apt-get install unrar -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.


In [1]:
!pip install --upgrade pip
!pip install numpy pandas pillow openpyxl tqdm arabic-reshaper python-bidi opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [arabic-reshaper]


In [12]:
import os
import random
import sys
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageEnhance, ImageFilter
import pandas as pd

img_h, img_w = 128, 256

fonts = [
    ("/content/NotoNastaliqUrdu-Medium.ttf", 60, "NotoNastaliq"),
    ("/content/Gulmarg Nataleeq_2013.ttf", 65, "Gulmarg"),
    ("/content/NaskhArabic.ttf", 70, "Naskh"),
    ("/content/Narqalam.ttf", 70, "Narqalam")
]

splits = {
    "train": pd.read_excel("/content/train.xlsx").iloc[:,0].dropna().astype(str).tolist(),
    "val": pd.read_excel("/content/val.xlsx").iloc[:,0].dropna().astype(str).tolist(),
    "test": pd.read_excel("/content/test.xlsx").iloc[:,0].dropna().astype(str).tolist()
}

def add_gaussian_blur(image, radius=0.3):
    return image.filter(ImageFilter.GaussianBlur(radius))

def add_yellow_tint(image, intensity=0.3):
    image_rgb = image.convert("RGB")
    yellow_layer = Image.new("RGB", image.size, (255, 255, 150))
    blended = Image.blend(image_rgb, yellow_layer, intensity)
    return blended.convert("L")

def add_bleed_through(image, alpha=0.04):
    h, w = image.height, image.width
    bleed = np.random.normal(200, 15, (h, w))
    bleed_img = Image.fromarray(np.clip(bleed, 0, 255).astype(np.uint8))
    return Image.blend(image, bleed_img, alpha)

def add_salt_and_pepper_noise(image, amount=0.001):
    arr = np.array(image)
    total = arr.size
    num_salt = int(total * amount / 2)
    num_pepper = int(total * amount / 2)

    coords_salt = [np.random.randint(0, i, num_salt) for i in arr.shape]
    coords_pepper = [np.random.randint(0, i, num_pepper) for i in arr.shape]

    arr[tuple(coords_salt)] = 255
    arr[tuple(coords_pepper)] = 0
    return Image.fromarray(arr)

def add_gaussian_noise(image, mean=0, sigma=10):
    arr = np.array(image).astype(np.float32)
    noise = np.random.normal(mean, sigma, arr.shape)
    noisy = np.clip(arr + noise, 0, 255)
    return Image.fromarray(noisy.astype(np.uint8))

def adjust_contrast(image, factor=None):
    if factor is None:
        factor = random.uniform(0.7, 1.3)
    enhancer = ImageEnhance.Contrast(image)
    return enhancer.enhance(factor)

def apply_vignette(image, strength=0.2):
    width, height = image.size
    x_center, y_center = width/2, height/2
    vignette = Image.new("L", (width, height), 0)
    for i in range(width):
        for j in range(height):
            distance = np.sqrt((i - x_center)**2 + (j - y_center)**2)
            factor = 1 - min(distance / (width/2), 1) * strength
            vignette.putpixel((i,j), int(factor*255))
    return Image.composite(image, Image.new("L", image.size, 255), vignette)

def apply_all_effects(image):
    img = add_gaussian_blur(image)
    img = add_yellow_tint(img)
    img = add_bleed_through(img)
    img = add_salt_and_pepper_noise(img)
    img = add_gaussian_noise(img)
    img = adjust_contrast(img)
    img = apply_vignette(img)
    return img

def render_word(word, font_path, font_size, output_path):
    font = ImageFont.truetype(font_path, font_size)
    img = Image.new("L", (img_w, img_h), 255)
    draw = ImageDraw.Draw(img)
    bbox = draw.textbbox((0,0), word, font=font)
    w, h = bbox[2]-bbox[0], bbox[3]-bbox[1]
    x = (img_w - w)//2
    y = (img_h - h)//2 - bbox[1]
    draw.text((x, y), word, font=font, fill=0)
    img = apply_all_effects(img)
    img.save(output_path)

output_root = "/content/VAADI_OCR_dataset_Noise"
os.makedirs(output_root, exist_ok=True)

for font_path, font_size, font_name in fonts:
    font_dir = os.path.join(output_root, font_name)
    os.makedirs(font_dir, exist_ok=True)
    for split_name, words in splits.items():
        split_dir = os.path.join(font_dir, split_name)
        os.makedirs(split_dir, exist_ok=True)
        for idx, word in enumerate(words, start=1):
            img_path = os.path.join(split_dir, f"{idx}.png")
            txt_path = os.path.join(split_dir, f"{idx}.txt")
            render_word(word, font_path, font_size, img_path)
            with open(txt_path, "w", encoding="utf-8") as f:
                f.write(word)

print("\nDataset generation complete")


Dataset generation complete
